# Episode 10 — Seeing Sound, and Back to Max

⏱ About 25 minutes · Difficulty ★★☆

**Musical element this episode controls: nothing new.** This episode is about **seeing clearly** what we built in Python and **taking it back to Max**.

**The Max equivalents:** `spectroscope~` (look), `read` on `buffer~` (load the wavetable), `read` on `dict` (load presets).

**What you will hear:** the three presets from Episode 8 — this time while looking at their spectrograms.

**You will end up with two files:**
- `export/wavetable.wav`: the 16-frame wavetable
- `export/presets.json`: the three presets

**By the end you can:**
- read a spectrum and a spectrogram
- recognise a filter envelope, a pitch change and a wavetable morph in a spectrogram
- export a wavetable and presets as files Max can read

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import os
import json
from scipy.io import wavfile

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # times must not be 0, or we would divide by zero below
    # while held: (0,0) → (a,1) → (a+d,s), then stays at s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # how high the envelope is at the moment of release (attack may not be finished)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # fall from that level to 0 over r seconds; np.clip stops it going negative
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # held before release, released after

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff may be one number or an array: make it an array as long as the sound
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- turn cutoff / res into the filter's internal coefficients (one set per sample) ---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping: higher res → smaller k → stronger resonance
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() turns arrays into plain lists: much faster to read one by one in a loop
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # two state variables = two history objects in gen~

    for n in range(n_samples):                 # one sample at a time, in order
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 is the lowpass output
        ic1 = 2 * v1 - ic1                     # update the state for the next sample
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = how far phase advances during one sample
    # cumsum adds the steps up; % 1 keeps the fractional part
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # MIDI 69 = 440 Hz; doubles every 12 semitones

def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # two envelopes
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator: the wt_pos knob plus what the mod env adds
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter: the cutoff knob, pushed up by cut_env octaves by the mod env
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

presets = {
    "pluck": dict(note=45, wt_pos=0.6, wt_env=0.0, cutoff=300, res=0.2, cut_env=4.0,
                  amp_a=0.005, amp_d=0.30, amp_s=0.0, amp_r=0.10,
                  mod_a=0.001, mod_d=0.15, mod_s=0.0, mod_r=0.10),
    "pad":   dict(note=57, wt_pos=0.2, wt_env=0.5, cutoff=800, res=0.1, cut_env=2.0,
                  amp_a=0.60, amp_d=0.30, amp_s=0.8, amp_r=0.70,
                  mod_a=1.00, mod_d=0.50, mod_s=0.6, mod_r=0.50),
    "wow":   dict(note=33, wt_pos=0.3, wt_env=0.7, cutoff=150, res=0.6, cut_env=4.5,
                  amp_a=0.01, amp_d=0.10, amp_s=0.9, amp_r=0.10,
                  mod_a=0.35, mod_d=0.40, mod_s=0.2, mod_r=0.20),
}

frames = make_frames()

n = sr                                         # 1 second
t = np.arange(n) / sr

## 1. The spectrum: a sound's "list of ingredients"

A waveform plot (`scope~`) shows **the amplitude at each moment**. It cannot answer "is this sound bright?".

A spectrum (`spectroscope~`) takes another view: **which frequencies are in this sound, and how loud is each.**

The operation that turns a sound into a spectrum is the FFT. How it computes is not covered here, only how to read it. The function below appeared in Episode 6.

- horizontal: frequency (Hz)
- vertical: level (dB). 0 = the loudest frequency; every 20 dB down is one tenth of the amplitude

### One sine

**Guess first:** what does the spectrum of a 440 Hz sine look like?

In [ ]:
def spectrum(x):
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))        # the strength of every frequency
    f = np.fft.rfftfreq(len(x), 1 / sr)                    # the matching frequency axis (Hz)
    return f, 20 * np.log10(X / np.max(X) + 1e-9)          # converted to dB, loudest = 0

def spec_plot(signals, max_freq=5000):
    plt.figure(figsize=(9, 3.5))
    for name in signals:
        f, db = spectrum(signals[name])
        plt.plot(f, db, label=name, linewidth=0.8)
    plt.xlim(0, max_freq)
    plt.ylim(-90, 5)
    plt.xlabel("frequency (Hz)")
    plt.ylabel("level (dB)")
    plt.legend()
    plt.show()

sine440 = np.sin(2 * np.pi * 440 * t)
spec_plot({"sine 440 Hz": sine440}, max_freq=2000)

Answer: a single spike at 440 Hz. A sine contains one frequency only — which is why it sounds "clean".

### Two sines added

Episode 1 built a chord by adding sines.

**You should see:** two spikes, at 440 and 660 Hz. The one at 660 is 6 dB shorter because its amplitude is half.

In [ ]:
two = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 660 * t)
spec_plot({"440 Hz + 660 Hz at half level": two}, max_freq=2000)

### Wavetable frames

The tables of Episode 3 were built by "stacking harmonics". The spectrum spreads those harmonics back out for you.

**You should see:**
- frame 0: one spike (110 Hz)
- frame 15: 32 evenly spaced spikes (110, 220, 330 ... up to 3520 Hz), getting shorter to the right

**"Dark" and "bright" on the plot simply mean: few spikes / many spikes.**

In [ ]:
phase = phasor(110, n)

spec_plot({
    "frame 15 (wt_pos = 1)": osc_wavetable(frames, phase, 1.0),
    "frame 0 (wt_pos = 0)": osc_wavetable(frames, phase, 0.0),
})

### What the filter did

**You should see:** after the lowpass the same row of spikes, with those above 600 Hz pushed down and a peak raised at 600 Hz by the resonance.

In [ ]:
raw = osc_wavetable(frames, phase, 1.0)

spec_plot({
    "raw": raw,
    "lowpass 600 Hz, res 0.8": lowpass(raw, 600, 0.8),
}, max_freq=4000)

## 2. The spectrogram: a spectrum that moves

A spectrum is a "photo": it squeezes the whole sound into one picture and cannot show **when** something happened.

A spectrogram is a "video": cut the sound into many short pieces, compute a spectrum for each and line them up side by side.

- horizontal: time
- vertical: frequency
- colour: brighter = louder

### A rising sine

**Guess first:** what does the spectrogram of a sine gliding from 100 Hz to 4000 Hz look like?

In [ ]:
def spectrogram(x, title="", max_freq=4000):
    plt.figure(figsize=(9, 3.5))
    # NFFT = length of each piece; noverlap = how much neighbouring pieces overlap
    with np.errstate(divide="ignore"):         # fully silent parts trigger a warning when converted to dB; ignore it here
        plt.specgram(x + 1e-9, NFFT=2048, Fs=sr, noverlap=1536, cmap="magma", vmin=-120)
    plt.ylim(0, max_freq)
    plt.xlabel("time (s)")
    plt.ylabel("frequency (Hz)")
    plt.title(title)
    plt.show()

glide = np.sin(2 * np.pi * phasor(np.geomspace(100, 4000, n), n))

spectrogram(glide, "sine gliding 100 -> 4000 Hz")
play(glide)

Answer: a bright line curving upward. It curves rather than running straight because we used `geomspace` (moving by ratio) while the plot's vertical axis is in Hz.

### The three presets

Now the three presets from Episode 8. **Listen first, then look, and find what you heard in the picture.**

**pluck — you should see:** a very narrow bright vertical band at the far left (the bright onset), then quickly only a few lines at the bottom, and black after 0.4 seconds.

In [ ]:
x = synth(presets["pluck"])
display(play(x))
spectrogram(x, "pluck")

**pad — you should see:** horizontal lines slowly "lighting up" from the left (the fade-in) while the bright region grows upward (brightening), then everything dimming after 1.2 seconds.

In [ ]:
x = synth(presets["pad"])
display(play(x))
spectrogram(x, "pad")

**wow — you should see:** a dense row of horizontal lines at the bottom (a low note's harmonics are close together). The bright region bulges upward, peaks around 0.35 seconds and settles back.

The "wow" you hear is that bulge. Its shape is the shape of the mod envelope.

In [ ]:
x = synth(presets["wow"])
display(play(x))
spectrogram(x, "wow", max_freq=3000)

### A quiz: guess the parameter from the picture

The three sounds below are one preset with **one** parameter changed each. Without listening, look at the pictures and guess what was changed in each:

- (a) `cut_env` set to 0
- (b) `note` raised one octave
- (c) `mod_a` set to 1.0

In [ ]:
base = presets["wow"]
quiz = {
    "A": dict(base, mod_a=1.0),
    "B": dict(base, cut_env=0.0),
    "C": dict(base, note=base["note"] + 12),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, name in zip(axes, quiz):
    with np.errstate(divide="ignore"):
        ax.specgram(synth(quiz[name]) + 1e-9, NFFT=2048, Fs=sr, noverlap=1536, cmap="magma", vmin=-120)
    ax.set_ylim(0, 3000)
    ax.set_title(name)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("frequency (Hz)")
plt.show()

<details><summary>Answer</summary>

- **A = (c)** `mod_a = 1.0`: the top of the bulge has moved to 1 second; it rises very slowly.
- **B = (a)** `cut_env = 0`: no bulge; the bright region stays at one height throughout (the filter does not move).
- **C = (b)** one octave up: the spacing between the horizontal lines has doubled.

</details>

Listen to confirm:

In [ ]:
for name in quiz:
    print(name)
    display(play(synth(quiz[name])))

## 3. Sending the wavetable back to Max

`frames` is a 16 × 2048 2-D array. A wav file is 1-D. So it has to be "flattened": frame 0, then frame 1, then frame 2 ...

`.reshape(-1)` does the flattening. A small example first:

In [ ]:
m = np.array([[1, 2, 3],
              [10, 20, 30]])

print("2-D:", m.shape)
print(m)
print("flattened:", m.reshape(-1))             # one row after another

This is the most common layout for wavetable files: 2048 samples per frame, end to end. Wavetable synths such as Serum can import it.

The file is written to an `export/` folder next to the notebook.

In [ ]:
os.makedirs("export", exist_ok=True)

flat = frames.reshape(-1).astype(np.float32)   # 16 × 2048 → one run of 32768 samples
wavfile.write("export/wavetable.wav", sr, 0.9 * flat)

print("frames:", frames.shape[0])
print("samples per frame:", frames.shape[1])
print("total samples in file:", len(flat))

### Verify

Load it back, take out frame 5 (samples 5 × 2048 to 6 × 2048) and compare with the original `frames[5]`.

In [ ]:
file_sr, loaded = wavfile.read("export/wavetable.wav")

k = 5
N = frames.shape[1]
frame_from_file = loaded[k * N : (k + 1) * N]          # where frame k sits in the file

print("frame 5 matches:", np.allclose(frame_from_file, 0.9 * frames[k], atol=1e-6))

### How to use it in Max

1. `buffer~ wt` → send it `read wavetable.wav`
2. patch `phasor~` into inlet 1 of `wave~ wt`
3. inlets 2 and 3 of `wave~` are the start and end of the range to read, **in milliseconds**

So "frame k" has to be converted to milliseconds:

- start = `k × 2048 / sr × 1000`
- end = `(k + 1) × 2048 / sr × 1000`

The cell below prints the start and end of every frame in milliseconds, ready to copy into Max. To crossfade between two frames, use two `wave~` objects reading neighbouring ranges and Episode 4's `(1 - mix) * a + mix * b` (in Max: `xfade~`, or a pair of `*~`).

In [ ]:
for k in range(frames.shape[0]):
    start_ms = k * N / sr * 1000
    end_ms = (k + 1) * N / sr * 1000
    print(f"frame {k:2d}:  {start_ms:8.3f} ms  to  {end_ms:8.3f} ms")

## 4. Sending presets back to Max

A preset is a `dict`. Episode 9 used JSON to store parameters. Max's `dict` object reads JSON files directly.

In [ ]:
with open("export/presets.json", "w") as f:
    json.dump(presets, f, indent=2)

print(json.dumps(presets["wow"], indent=2))    # this is what the "wow" section of the file looks like

### How to use it in Max

1. `dict presets` → send it `read presets.json`
2. send `get wow::cutoff` → the `dict` outputs that value (150). `::` means "one level down"
3. use `route` to send the retrieved values to the `param` objects of the same name in your RNBO patch

As long as the synth on the Max side uses **the same parameter names and the same signal flow** (the diagram from Episode 8), one preset should give the same sound in both places.

In practice the filter and envelope curves on the two sides will not be identical (`lores~` and our `lowpass()` are different algorithms), so the result will be **very close but not the same**. For an exact match you would write the arithmetic of Episode 6 out in gen~, line for line.

## 5. The whole series at a glance

| Ep. | Max / RNBO | Python |
|---|---|---|
| 1 | `cycle~`, `dac~`, `scope~`, signals | `np.sin`, `play`, `scope`, arrays |
| 2 | `phasor~`, `<~`, `abs~` | `(freq * t) % 1`, `np.where`, `np.abs` |
| 3 | `buffer~`, `index~`, `wave~` | `table[idx]`, linear interpolation |
| 4 | WT POS, `xfade~`, `line~` | 2-D `frames`, `np.linspace` |
| 5 | `adsr~`, `function`, `*~` | `np.interp`, `osc * env` |
| 6 | `onepole~`, `lores~`, gen~ `history`, `noise~` | a `for` loop + state variables |
| 7 | signals into parameter inlets, LFO, signal input of `phasor~` | arrays as parameters, `np.cumsum` |
| 8 | RNBO patch, `param`, `dict`, `mtof` | a function, a `dict` |
| 9 | `random`, `sfrecord~` (awkward) | `rng` + a loop + saving files |
| 10 | `spectroscope~`, `buffer~ read`, `dict read` | `np.fft`, `specgram`, `wavfile`, `json` |

## Three things to take away

1. **In Python, sound is an array.** Anything without feedback is computed all at once; anything with feedback (filters, phase accumulation) has to "remember the last time".
2. **Parameters and signals are the same kind of thing.** Pass one number for static, pass an array for modulation.
3. **Python handles offline and bulk work; Max handles real time and interaction.** With shared parameter names they can hand things to each other.

## Exercises

**1.** Plot the spectrum of Episode 2's naive saw (`2 * phase - 1`) at 2500 Hz with the horizontal axis up to 22050 Hz. Find the lines that are not at whole multiples of 2500 — that is aliasing.

<details><summary>Answer</summary>

```python
ph = phasor(2500, n)
spec_plot({"naive saw 2500 Hz": 2 * ph - 1}, max_freq=22050)
```
2500, 5000, 7500 ... are the real harmonics. The shorter lines between them are the folded-back ones.
</details>

**2.** Plot the spectrogram of Episode 7's vibrato (220 Hz, 5 per second, half a semitone each way), showing only 0–1500 Hz. **Guess first:** what shape are the lines?

<details><summary>Answer</summary>

```python
vib = 220 * 2 ** (0.5 / 12 * np.sin(2 * np.pi * 5 * t))
spectrogram(osc_wavetable(frames, phasor(vib, n), 0.8), "vibrato", max_freq=1500)
```
Every harmonic is a wavy line; higher harmonics wave more widely (their frequencies are whole multiples of the fundamental, so the wobble is multiplied by the same factor).
</details>

**3.** Export the "sine to square" frames from Episode 4's exercise (or any frames of your own) as `export/my_wavetable.wav`.

<details><summary>Answer</summary>

```python
sq = []
for f in range(16):
    amps = []
    for k in range(1, 2 * f + 2):
        amps.append(1 / k if k % 2 == 1 else 0)
    sq.append(make_table(amps))
sq = np.array(sq)
wavfile.write("export/my_wavetable.wav", sr, (0.9 * sq.reshape(-1)).astype(np.float32))
```
</details>

## Where to go next

- swap the envelopes for exponential curves (the idea from Episode 5, section 7)
- add a second oscillator, detune, noise
- use the `sounddevice` library for real-time output from Python
- rebuild this synth in gen~ / RNBO following Episode 8's signal flow and load the two files exported here
- train a model on Episode 9's dataset: listen to a sound → predict its parameters